## Imports + warnings

In [1]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMClassifier was fitted with feature names"
)

from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    log_loss, r2_score, classification_report
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## Configs + Load CSVs

In [2]:
RANDOM_SEED = 42

# Toggle: run with all classes or drop some (for A/B comparison)
DROP_CLASSES = True
DROP_WEAK_LINKS = {
    "LeftHeelRises",
    "RightHeelRises",
    "LeftKneeMovesInward",
    "RightAsymmetricalWeightShift",
}

# How to handle ties (multiple indicators share max)
DROP_TIES = True

# Update paths if needed
SCORES_PATH = "../../data/processed/20190108_scores_and_weak_links.csv"
FEATURES_PATH = "../../data/processed/AimoScore_WeakLink_big_scores.csv"

scores = pd.read_csv(SCORES_PATH)
big = pd.read_csv(FEATURES_PATH)

print("scores:", scores.shape)
print("big:", big.shape)
display(scores.head(2))
display(big.head(2))

scores: (2096, 17)
big: (2094, 43)


,ID,Date,SCORE,ForwardHead,LeftArmFallForward,RightArmFallForward,LeftShoulderElevation,RightShoulderElevation,ExcessiveForwardLean,LeftAsymmetricalWeightShift,RightAsymmetricalWeightShift,LeftKneeMovesInward,RightKneeMovesInward,LeftKneeMovesOutward,RightKneeMovesOutward,LeftHeelRises,RightHeelRises
0,cf0e95ab-d6ec-475a-9cc0-c9444b8dd080.Kinect,2018-07-19,0.612532,0.31,1.34,1.15,0.33,0.49,0.00,0.0,0.08,0.0,0.0,0.0,0.77,0.0,0.0
1,6210cc6d-575d-4e9f-bab1-4fe1da2a30a3.Kinect,2018-07-30,0.388809,1.49,1.42,1.39,0.00,0.00,1.08,0.1,0.00,0.0,0.0,0.0,0.40,0.0,0.0


,AimoScore,No_1_Angle_Deviation,No_2_Angle_Deviation,No_3_Angle_Deviation,No_4_Angle_Deviation,No_5_Angle_Deviation,No_6_Angle_Deviation,No_7_Angle_Deviation,No_8_Angle_Deviation,No_9_Angle_Deviation,...,No_20_NASM_Deviation,No_21_NASM_Deviation,No_22_NASM_Deviation,No_23_NASM_Deviation,No_24_NASM_Deviation,No_25_NASM_Deviation,No_1_Time_Deviation,No_2_Time_Deviation,EstimatedScore,ID
0,0.323667,0.538020,0.815878,0.346724,0.382114,0.302248,0.947872,0.275945,0.52176,0.457198,...,0.656624,0.642276,0.552846,0.648972,0.578192,0.560019,0.821616,0.818747,0.209947,0003cdcc-86ed-494a-a3b5-90d09e96e06b.Kinect
1,0.323699,0.443807,0.306552,0.823529,0.188905,0.497370,0.140124,0.664275,0.52176,0.729316,...,0.721186,0.826399,0.805356,0.848876,0.889048,0.816834,0.307987,0.248207,0.457198,003115c4-bdb8-491c-b571-8fcebdecf8ed.Kinect


## Build weakest-link labels (drop classes + drop ties)

In [3]:
NON_INDICATORS = {"ID", "Date", "SCORE"}
indicator_cols = [c for c in scores.columns if c not in NON_INDICATORS]

if DROP_CLASSES:
    indicator_cols = [c for c in indicator_cols if c not in DROP_WEAK_LINKS]

scores[indicator_cols] = scores[indicator_cols].apply(pd.to_numeric, errors="coerce")

row_max = scores[indicator_cols].max(axis=1)
ties = (scores[indicator_cols].eq(row_max, axis=0).sum(axis=1) > 1)

scores["WeakestLink"] = scores[indicator_cols].idxmax(axis=1)
scores["HasTie"] = ties

if DROP_TIES:
    scores_labeled = scores.loc[~scores["HasTie"]].copy()
else:
    scores_labeled = scores.copy()

scores_labeled = scores_labeled.dropna(subset=["WeakestLink"])

print("Indicator classes used:", len(indicator_cols))
print("Tie rows:", int(scores["HasTie"].sum()))
print("Rows kept:", scores_labeled.shape[0])
print("Unique classes:", scores_labeled["WeakestLink"].nunique())
display(scores_labeled["WeakestLink"].value_counts())


Indicator classes used: 10
Tie rows: 372
Rows kept: 1724
Unique classes: 10


WeakestLink
RightArmFallForward            454
LeftArmFallForward             327
RightKneeMovesOutward          279
RightShoulderElevation         252
ExcessiveForwardLean            99
ForwardHead                     96
LeftAsymmetricalWeightShift     71
LeftShoulderElevation           51
LeftKneeMovesOutward            49
RightKneeMovesInward            46
Name: count, dtype: int64

## Merge features + build X/y (+ drop NaNs)

In [4]:
labels = scores_labeled[["ID", "WeakestLink"]]
data = big.merge(labels, on="ID", how="inner")

print("After merge:", data.shape)
print("Unique classes:", data["WeakestLink"].nunique())

angle_cols = [c for c in data.columns if c.startswith("No_") and c.endswith("_Angle_Deviation")]
nasm_cols  = [c for c in data.columns if c.startswith("No_") and c.endswith("_NASM_Deviation")]
feature_cols = sorted(angle_cols) + sorted(nasm_cols)

print("Angle features:", len(angle_cols))
print("NASM features:", len(nasm_cols))
print("Total features:", len(feature_cols))

X = data[feature_cols].copy()
y = data["WeakestLink"].astype(str).copy()

X = X.apply(pd.to_numeric, errors="coerce")
mask = X.notna().all(axis=1)

X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print("After dropping NaNs:", X.shape, y.shape)
print("Min class count:", y.value_counts().min())


After merge: (1724, 44)
Unique classes: 10
Angle features: 13
NASM features: 25
Total features: 38
After dropping NaNs: (1724, 38) (1724,)
Min class count: 46


## Region mappings (2-way and 3-way)

In [5]:
UPPER = {
    "ForwardHead",
    "LeftArmFallForward",
    "RightArmFallForward",
    "LeftShoulderElevation",
    "RightShoulderElevation",
}
TRUNK = {"ExcessiveForwardLean"}

def to_region2(lbl: str) -> str:
    return "Upper" if lbl in UPPER else "Lower"

def to_region3(lbl: str) -> str:
    if lbl in UPPER: return "Upper"
    if lbl in TRUNK: return "Trunk"
    return "Lower"

region2 = y.map(to_region2)
region3 = y.map(to_region3)

print("2-way region counts:")
print(region2.value_counts())
print("\n3-way region counts:")
print(region3.value_counts())


2-way region counts:
WeakestLink
Upper    1181
Lower     543
Name: count, dtype: int64

3-way region counts:
WeakestLink
Upper    1181
Lower     444
Trunk      99
Name: count, dtype: int64


## Metrics helpers (includes weighted F1)

In [20]:
def align_proba_to_all_classes(proba, model_classes, all_classes, renormalize=True, eps=1e-12):
    """
    Expand/reorder proba columns so they match all_classes.
    Missing classes get probability 0.

    If renormalize=True, rows are scaled to sum to 1 to keep log_loss happy.
    """
    model_classes = list(model_classes)
    all_classes = list(all_classes)

    out = np.zeros((proba.shape[0], len(all_classes)), dtype=float)
    idx = {c: i for i, c in enumerate(all_classes)}

    for j, c in enumerate(model_classes):
        out[:, idx[c]] = proba[:, j]

    if renormalize:
        row_sums = out.sum(axis=1, keepdims=True)
        row_sums = np.maximum(row_sums, eps)
        out = out / row_sums

    return out

def multiclass_r2_from_proba(y_true, proba_aligned, classes):
    cls_to_idx = {c:i for i, c in enumerate(classes)}
    y_idx = np.array([cls_to_idx[t] for t in y_true])
    y_onehot = np.zeros((len(y_true), len(classes)), dtype=float)
    y_onehot[np.arange(len(y_true)), y_idx] = 1.0
    return r2_score(y_onehot, proba_aligned, multioutput="uniform_average")

def summarize_metrics(y_true, y_pred, proba_aligned, all_classes):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "log_loss": log_loss(y_true, proba_aligned, labels=list(all_classes)),
    }

## Universal “fit/predict_proba” wrapper (handles XGB + CatBoost)

In [21]:
def _final_estimator(model):
    return model.steps[-1][1] if isinstance(model, Pipeline) else model

def _is_xgb(model):
    return isinstance(_final_estimator(model), XGBClassifier)

def fit_predict_proba_with_optional_encoding(model, Xtr, ytr, Xte, all_classes):
    """
    Returns:
      y_pred_str (np.array of class names)
      proba_aligned (ndarray aligned to all_classes)
    Handles XGBoost label encoding reliably. CatBoost usually handles strings, but we keep it simple.
    """
    m = clone(model)

    if _is_xgb(m):
        le = LabelEncoder()
        le.fit(list(all_classes))          # stable mapping
        ytr_enc = le.transform(ytr)

        m.fit(Xtr, ytr_enc)

        proba_raw = m.predict_proba(Xte)
        ypred_enc = m.predict(Xte)
        ypred_str = le.inverse_transform(ypred_enc)

        # proba columns correspond to le.classes_
        proba_aligned = align_proba_to_all_classes(proba_raw, le.classes_, all_classes)
        return ypred_str, proba_aligned

    else:
        # Most sklearn models + CatBoost can fit on string labels
        m.fit(Xtr, ytr)

        proba_raw = m.predict_proba(Xte)
        classes = _final_estimator(m).classes_ if hasattr(_final_estimator(m), "classes_") else m.classes_
        proba_aligned = align_proba_to_all_classes(proba_raw, classes, all_classes)

        ypred_str = m.predict(Xte)
        return np.array(ypred_str), proba_aligned


## CV setup

In [22]:
all_classes = sorted(y.unique())

min_class_count = int(y.value_counts().min())
n_splits = min(5, min_class_count)
if n_splits < 2:
    raise ValueError(f"Not enough samples for CV. Min class count = {min_class_count}")

print("Min class count:", min_class_count)
print("Using n_splits:", n_splits)

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)


Min class count: 46
Using n_splits: 5


## Define baseline models

In [23]:
baseline_models = {
    "logreg_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
    ]),

    "rf_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=800,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),

    "lightgbm_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),

    "xgb_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=700,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),
}


## Baseline CV evaluation

In [24]:
baseline_results = []
baseline_details = {}

for name, model in baseline_models.items():
    fold_rows = []
    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]

        ypred, proba = fit_predict_proba_with_optional_encoding(model, Xtr, ytr, Xte, all_classes)
        metrics = summarize_metrics(yte, ypred, proba, all_classes)
        metrics["fold"] = fold
        fold_rows.append(metrics)

    df = pd.DataFrame(fold_rows)
    baseline_details[name] = df

    mean_row = df.mean(numeric_only=True).to_dict()
    mean_row["model"] = name
    baseline_results.append(mean_row)

baseline_summary = (
    pd.DataFrame(baseline_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(baseline_summary)


,accuracy,macro_f1,weighted_f1,log_loss,fold,model
0,0.658343,0.556746,0.650656,1.652920,3.0,lightgbm_balanced
1,0.657765,0.541338,0.646316,1.070890,3.0,xgb_balanced
2,0.623554,0.498532,0.599544,1.096200,3.0,rf_balanced
3,0.541164,0.484739,0.553784,1.273732,3.0,logreg_balanced


## Stage 1 (region2) models + CV

In [25]:
stage1_models = {
    "s1_logreg": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
    ]),
    "s1_lightgbm": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
    "s1_xgb": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),
}

stage1_rows = []
for name, model in stage1_models.items():
    fold_rows = []
    for fold, (tr, te) in enumerate(skf.split(X, region2), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        rtr, rte = region2.iloc[tr], region2.iloc[te]

        # Stage1 is binary labels "Upper"/"Lower" – XGB can handle these fine, but we'll keep consistent:
        ypred, proba = fit_predict_proba_with_optional_encoding(model, Xtr, rtr, Xte, sorted(region2.unique()))
        metrics = {
            "accuracy": accuracy_score(rte, ypred),
            "balanced_accuracy": balanced_accuracy_score(rte, ypred),
            "macro_f1": f1_score(rte, ypred, average="macro"),
            "weighted_f1": f1_score(rte, ypred, average="weighted"),
            "fold": fold
        }
        fold_rows.append(metrics)

    df = pd.DataFrame(fold_rows)
    mean_row = df.mean(numeric_only=True).to_dict()
    mean_row["model"] = name
    stage1_rows.append(mean_row)

stage1_summary = pd.DataFrame(stage1_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
display(stage1_summary)


,accuracy,balanced_accuracy,macro_f1,weighted_f1,fold,model
0,0.875295,0.830426,0.847048,0.871350,3.0,s1_xgb
1,0.863687,0.819965,0.833764,0.859825,3.0,s1_lightgbm
2,0.794087,0.783596,0.770924,0.797779,3.0,s1_logreg


## Stage 2 expert models (Upper/Lower)

In [26]:
stage2_upper_models = {
    "upper_lgbm": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=700,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
    "upper_xgb": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=800,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),
}

stage2_lower_models = {
    "lower_lgbm": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=900,
            learning_rate=0.03,
            num_leaves=63,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
    "lower_xgb": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=900,
            learning_rate=0.04,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),
}


## 2-way hierarchical CV (soft/hard) with XGB-safe training

In [27]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

def _final_estimator(m):
    return m.steps[-1][1] if isinstance(m, Pipeline) else m

def _is_xgb(m):
    return _final_estimator(m).__class__.__name__ == "XGBClassifier"


def _fit_predict_proba(m, Xtr, ytr, Xte, *, all_classes=None, encoder_mode="present"):
    """
    Fits model m on (Xtr, ytr), returns:
      (ypred_labels, proba_aligned, model_classes_labels)

    If model is XGB:
      - encoder_mode="present": encode using classes present in ytr (contiguous 0..K-1)  ✅ for stage2
      - encoder_mode="fixed":   encode using all_classes (stable mapping)               ✅ for stage1

    If all_classes is not None:
      aligns probability columns to all_classes.
    """
    m = clone(m)

    if _is_xgb(m):
        le = LabelEncoder()

        if encoder_mode == "fixed":
            if all_classes is None:
                raise ValueError("encoder_mode='fixed' requires all_classes.")
            le.fit(np.array(all_classes))
        elif encoder_mode == "present":
            le.fit(np.array(sorted(pd.unique(ytr))))
        else:
            raise ValueError("encoder_mode must be 'present' or 'fixed'")

        ytr_enc = le.transform(ytr)
        m.fit(Xtr, ytr_enc)

        proba_raw = m.predict_proba(Xte)
        ypred_enc = m.predict(Xte)
        ypred = le.inverse_transform(ypred_enc)
        model_classes = list(le.classes_)

    else:
        m.fit(Xtr, ytr)
        proba_raw = m.predict_proba(Xte)
        fe = _final_estimator(m)
        model_classes = list(fe.classes_) if hasattr(fe, "classes_") else list(sorted(pd.unique(ytr)))
        ypred = m.predict(Xte)

    if all_classes is not None:
        proba_aligned = align_proba_to_all_classes(proba_raw, model_classes, all_classes)
        return ypred, proba_aligned, model_classes

    return ypred, proba_raw, model_classes


def hierarchical2_cv(stage1, upper_model, lower_model,
                     X, y, region2, skf, all_classes, routing="soft"):
    """
    2-way hierarchical CV (Upper/Lower experts).
    Handles XGB label requirements:
      - Stage 1 (Upper/Lower): encoder_mode='fixed' to keep stable order
      - Stage 2 experts:       encoder_mode='present' to keep labels contiguous
    """
    rows = []
    all_classes = list(all_classes)

    # fixed order for stage1 proba alignment
    region_classes = ["Lower", "Upper"]

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        rtr, rte = region2.iloc[tr], region2.iloc[te]

        # ---------- Stage 1 ----------
        s1 = clone(stage1)
        _, s1_proba, _ = _fit_predict_proba(
            s1, Xtr, rtr, Xte,
            all_classes=region_classes,
            encoder_mode="fixed"   # ✅ stable mapping Lower/Upper even if XGB
        )

        p_lower = s1_proba[:, region_classes.index("Lower")]
        p_upper = s1_proba[:, region_classes.index("Upper")]

        # ---------- Stage 2 train experts on TRUE region ----------
        Xtr_u = Xtr.loc[rtr == "Upper"]
        ytr_u = ytr.loc[rtr == "Upper"]
        Xtr_l = Xtr.loc[rtr == "Lower"]
        ytr_l = ytr.loc[rtr == "Lower"]

        if len(Xtr_u) == 0 or len(Xtr_l) == 0:
            continue

        u = clone(upper_model)
        l = clone(lower_model)

        # experts predict probabilities for ALL test rows, aligned to all_classes
        _, pu, _ = _fit_predict_proba(
            u, Xtr_u, ytr_u, Xte,
            all_classes=all_classes,
            encoder_mode="present"  # ✅ contiguous labels for region subset
        )
        _, pl, _ = _fit_predict_proba(
            l, Xtr_l, ytr_l, Xte,
            all_classes=all_classes,
            encoder_mode="present"  # ✅ contiguous labels for region subset
        )

        # ---------- Combine ----------
        full_proba = np.zeros((len(Xte), len(all_classes)), dtype=float)

        if routing == "soft":
            full_proba += (p_upper[:, None] * pu)
            full_proba += (p_lower[:, None] * pl)

        elif routing == "hard":
            pick_upper = (p_upper >= p_lower)
            full_proba[pick_upper] = pu[pick_upper]
            full_proba[~pick_upper] = pl[~pick_upper]
        else:
            raise ValueError("routing must be 'soft' or 'hard'")

        # renormalize for log_loss sanity
        row_sums = full_proba.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        full_proba = full_proba / row_sums

        ypred = np.array(all_classes)[full_proba.argmax(axis=1)]

        metrics = summarize_metrics(yte, ypred, full_proba, all_classes)
        metrics["fold"] = fold
        rows.append(metrics)

    return pd.DataFrame(rows)


## Run hierarchical combinations (soft) + best + soft vs hard

In [28]:
hier_results = []
hier_details = {}

for s1_name, s1 in stage1_models.items():
    for u_name, u in stage2_upper_models.items():
        for l_name, l in stage2_lower_models.items():
            key = f"{s1_name}__{u_name}__{l_name}"
            df = hierarchical2_cv(s1, u, l, X, y, region2, skf, all_classes, routing="soft")
            hier_details[key] = df
            mean_row = df.mean(numeric_only=True).to_dict()
            mean_row["model"] = key
            hier_results.append(mean_row)

hier_summary = (
    pd.DataFrame(hier_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)
display(hier_summary)

best_hier_key = hier_summary.iloc[0]["model"]
best_parts = best_hier_key.split("__")
best_s1 = stage1_models[best_parts[0]]
best_u  = stage2_upper_models[best_parts[1]]
best_l  = stage2_lower_models[best_parts[2]]

soft_df = hierarchical2_cv(best_s1, best_u, best_l, X, y, region2, skf, all_classes, routing="soft")
hard_df = hierarchical2_cv(best_s1, best_u, best_l, X, y, region2, skf, all_classes, routing="hard")

print("Best hierarchical:", best_hier_key)
print("\nSOFT routing mean:")
display(soft_df.mean(numeric_only=True))
print("\nHARD routing mean:")
display(hard_df.mean(numeric_only=True))


,accuracy,macro_f1,weighted_f1,log_loss,fold,model
0,0.648480,0.527797,0.636111,1.365599,3.0,s1_lightgbm__upper_xgb__lower_lgbm
1,0.649638,0.527470,0.637100,1.223305,3.0,s1_xgb__upper_xgb__lower_lgbm
2,0.649061,0.525501,0.636209,1.264410,3.0,s1_lightgbm__upper_xgb__lower_xgb
3,0.648475,0.519589,0.634553,1.122116,3.0,s1_xgb__upper_xgb__lower_xgb
4,0.638620,0.515878,0.627418,1.626683,3.0,s1_xgb__upper_lgbm__lower_lgbm
5,0.640362,0.514227,0.628636,1.768977,3.0,s1_lightgbm__upper_lgbm__lower_lgbm
6,0.639200,0.512173,0.626765,1.525493,3.0,s1_xgb__upper_lgbm__lower_xgb
7,0.640944,0.511659,0.628656,1.667787,3.0,s1_lightgbm__upper_lgbm__lower_xgb
8,0.599183,0.459315,0.587440,1.224568,3.0,s1_logreg__upper_xgb__lower_xgb
9,0.593377,0.456198,0.584297,1.729135,3.0,s1_logreg__upper_lgbm__lower_lgbm


Best hierarchical: s1_lightgbm__upper_xgb__lower_lgbm

SOFT routing mean:


accuracy       0.648480
macro_f1       0.527797
weighted_f1    0.636111
log_loss       1.365599
fold           3.000000
dtype: float64


HARD routing mean:


accuracy       0.644998
macro_f1       0.517939
weighted_f1    0.632210
log_loss       5.250530
fold           3.000000
dtype: float64

## 3-way hierarchical (soft mixture)

In [29]:
def hierarchical3_cv(stage1, expert_models, X, y, region3, skf, all_classes):
    rows = []
    all_classes = list(all_classes)
    cls_to_col = {c: i for i, c in enumerate(all_classes)}
    regions = ["Upper", "Trunk", "Lower"]

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        rtr, rte = region3.iloc[tr], region3.iloc[te]

        # Stage 1 region classifier (3-way)
        s1 = clone(stage1)
        s1.fit(Xtr, rtr)
        s1_proba = s1.predict_proba(Xte)
        s1_classes = list(_final_estimator(s1).classes_)

        full_proba = np.zeros((len(Xte), len(all_classes)), dtype=float)

        for reg in regions:
            # train expert on TRUE region rows only
            Xr = Xtr.loc[rtr == reg]
            yr = ytr.loc[rtr == reg]
            reg_classes = sorted(yr.unique())

            # If region has no samples (shouldn't happen here), skip
            if len(reg_classes) == 0:
                continue

            # mixture weight P(reg)
            p_reg = s1_proba[:, s1_classes.index(reg)]

            # conditional proba P(class | reg) aligned to reg_classes
            _, pr = fit_predict_proba_with_optional_encoding(expert_models[reg], Xr, yr, Xte, reg_classes)

            for j, c in enumerate(reg_classes):
                full_proba[:, cls_to_col[c]] += p_reg * pr[:, j]

        # Renormalize
        row_sums = full_proba.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        full_proba = full_proba / row_sums

        ypred = np.array(all_classes)[full_proba.argmax(axis=1)]
        metrics = summarize_metrics(yte, ypred, full_proba, all_classes)
        metrics["fold"] = fold
        rows.append(metrics)

    return pd.DataFrame(rows)


stage1_3way = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", LGBMClassifier(
        n_estimators=450,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1
    ))
])

experts_3way = {
    "Upper": stage2_upper_models["upper_lgbm"],
    "Trunk": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=350,
            learning_rate=0.05,
            num_leaves=15,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
    "Lower": stage2_lower_models["lower_lgbm"],
}

hier3_df = hierarchical3_cv(stage1_3way, experts_3way, X, y, region3, skf, all_classes)

print("3-way hierarchical mean:")
display(hier3_df.mean(numeric_only=True))
display(hier3_df)


3-way hierarchical mean:


accuracy       0.638045
macro_f1       0.510955
weighted_f1    0.625369
log_loss       1.913080
fold           3.000000
dtype: float64

,accuracy,macro_f1,weighted_f1,log_loss,fold
0,0.614493,0.470915,0.598183,2.169428,1
1,0.684058,0.545596,0.671802,1.629425,2
2,0.637681,0.523809,0.627422,1.918759,3
3,0.626087,0.500074,0.612837,1.881629,4
4,0.627907,0.514380,0.616599,1.966157,5


## Best baseline vs best hierarchical comparison

In [30]:
best_base = baseline_summary.iloc[0]["model"]
base_mean = baseline_details[best_base].mean(numeric_only=True)
hier_mean = hier_details[best_hier_key].mean(numeric_only=True)

compare = pd.DataFrame({
    "baseline_best_mean": base_mean,
    "hierarchical_best_mean": hier_mean
})
compare["delta(hier-base)"] = compare["hierarchical_best_mean"] - compare["baseline_best_mean"]
display(compare)

print("Best baseline:", best_base)
print("Best hierarchical:", best_hier_key)


,baseline_best_mean,hierarchical_best_mean,delta(hier-base)
accuracy,0.658343,0.648480,-0.009863
macro_f1,0.556746,0.527797,-0.028950
weighted_f1,0.650656,0.636111,-0.014545
log_loss,1.652920,1.365599,-0.287321
fold,3.000000,3.000000,0.000000


Best baseline: lightgbm_balanced
Best hierarchical: s1_lightgbm__upper_xgb__lower_lgbm


## Per-class report (holdout) with XGB-safe handling

In [31]:
Xtr, Xte, ytr, yte, rtr, rte = train_test_split(
    X, y, region2, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

# --- Baseline best ---
base_model = baseline_models[best_base]
ypred_base, _ = fit_predict_proba_with_optional_encoding(base_model, Xtr, ytr, Xte, all_classes)

print("BASELINE (best):", best_base)
print(classification_report(yte, ypred_base, zero_division=0))

# --- Hierarchical best (soft mixture) ---
# Fit stage1 on region
s1 = clone(best_s1)
s1.fit(Xtr, rtr)
p = s1.predict_proba(Xte)
s1_classes = list(_final_estimator(s1).classes_)
p_upper = p[:, s1_classes.index("Upper")]
p_lower = p[:, s1_classes.index("Lower")]

# Train experts
Xu_tr = Xtr.loc[rtr == "Upper"]
yu_tr = ytr.loc[rtr == "Upper"]
Xl_tr = Xtr.loc[rtr == "Lower"]
yl_tr = ytr.loc[rtr == "Lower"]

upper_classes = sorted(yu_tr.unique())
lower_classes = sorted(yl_tr.unique())

_, pu = fit_predict_proba_with_optional_encoding(best_u, Xu_tr, yu_tr, Xte, upper_classes)
_, pl = fit_predict_proba_with_optional_encoding(best_l, Xl_tr, yl_tr, Xte, lower_classes)

full = np.zeros((len(Xte), len(all_classes)))
col = {c:i for i, c in enumerate(all_classes)}

for j, c in enumerate(upper_classes):
    full[:, col[c]] += p_upper * pu[:, j]
for j, c in enumerate(lower_classes):
    full[:, col[c]] += p_lower * pl[:, j]

full = full / np.maximum(full.sum(axis=1, keepdims=True), 1e-12)
hier_pred = np.array(all_classes)[full.argmax(axis=1)]

print("HIERARCHICAL (best soft):", best_hier_key)
print(classification_report(yte, hier_pred, zero_division=0))


BASELINE (best): lightgbm_balanced
                             precision    recall  f1-score   support

       ExcessiveForwardLean       0.58      0.75      0.65        20
                ForwardHead       0.62      0.68      0.65        19
         LeftArmFallForward       0.57      0.58      0.57        66
LeftAsymmetricalWeightShift       0.59      0.71      0.65        14
       LeftKneeMovesOutward       0.75      0.60      0.67        10
      LeftShoulderElevation       0.29      0.20      0.24        10
        RightArmFallForward       0.65      0.70      0.67        91
       RightKneeMovesInward       0.33      0.11      0.17         9
      RightKneeMovesOutward       1.00      0.95      0.97        56
     RightShoulderElevation       0.73      0.64      0.68        50

                   accuracy                           0.68       345
                  macro avg       0.61      0.59      0.59       345
               weighted avg       0.68      0.68      0.67       3

In [32]:
# === Number of classes seen during training ===

n_classes_total = y.nunique()
n_classes_train = ytr.nunique()
n_classes_test = yte.nunique()

print(f"Total classes in dataset: {n_classes_total}")
print(f"Classes present in training set: {n_classes_train}")
print(f"Classes present in test set: {n_classes_test}")



Total classes in dataset: 10
Classes present in training set: 10
Classes present in test set: 10


In [33]:
# === Class coverage: full dataset vs train vs test ===

coverage_df = pd.DataFrame({
    "full_dataset": y.value_counts(),
    "training_set": ytr.value_counts(),
    "test_set": yte.value_counts(),
}).fillna(0).astype(int)

coverage_df["present_in_train"] = coverage_df["training_set"] > 0
coverage_df["present_in_test"] = coverage_df["test_set"] > 0

coverage_df.sort_values("full_dataset")


,full_dataset,training_set,test_set,present_in_train,present_in_test
WeakestLink,,,,,
RightKneeMovesInward,46,37,9,True,True
LeftKneeMovesOutward,49,39,10,True,True
LeftShoulderElevation,51,41,10,True,True
LeftAsymmetricalWeightShift,71,57,14,True,True
ForwardHead,96,77,19,True,True
ExcessiveForwardLean,99,79,20,True,True
RightShoulderElevation,252,202,50,True,True
RightKneeMovesOutward,278,222,56,True,True
LeftArmFallForward,328,262,66,True,True
